# 오류가 포함된 테스트 분개장 생성

정상 분개장 일부에 의도적으로 오류를 삽입하고
오류 검증 프로그램의 성능을 확인하기 위한 정답표를 생성한다.

## 1. 정상 분개장 불러오기

In [1]:
# 데이터 처리와 경로 설정에 필요한 라이브러리
import pandas as pd
from pathlib import Path


# 현재 실행 위치를 기준으로 프로젝트 루트 설정
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 정상 분개장 파일 불러오기
sample_path = project_root / "data" / "raw" / "journal_sample.xlsx"

normal_journal = pd.read_excel(
    sample_path,
    sheet_name="분개장"
)

print("불러온 거래 수:", len(normal_journal))
normal_journal.head()

불러온 거래 수: 10


,voucher_id,transaction_date,department_code,partner_code,evidence_type,evidence_no,description,debit_account_1,debit_amount_1,debit_account_2,debit_amount_2,credit_account_1,credit_amount_1,credit_account_2,credit_amount_2,supply_amount,vat_amount,total_amount,remarks
0,JV202608001,2026-08-01,D002,V001,전자세금계산서,TAX-202608-001,고무 원재료 외상 매입,1210,5000000,1180.0,500000,2110,5500000,NaN,0,5000000,500000,5500000,8월 원재료 구매
1,JV202608002,2026-08-02,D003,V003,법인카드,CARD-202608-001,생산 작업용 장갑 구매,5130,200000,1180.0,20000,2140,220000,NaN,0,200000,20000,220000,생산1팀 사용
2,JV202608003,2026-08-03,D007,V004,전자세금계산서,TAX-202608-002,제품 포장재 외상 구매,5130,800000,1180.0,80000,2110,880000,NaN,0,800000,80000,880000,출하용 포장재
3,JV202608004,2026-08-05,D007,V005,전자세금계산서,TAX-202608-003,제품 납품 운송비,5140,600000,1180.0,60000,2120,660000,NaN,0,600000,60000,660000,8월 1주차 운송비
4,JV202608005,2026-08-08,D003,V006,전자세금계산서,TAX-202608-004,공장 전기요금,5150,1500000,1180.0,150000,2120,1650000,NaN,0,1500000,150000,1650000,생산공장 전력 사용


## 2. 테스트용 오류 삽입

In [2]:
# 정상 데이터를 보존하기 위해 별도의 복사본 생성
error_journal = normal_journal.copy()

# 1번 거래: 기준표에 없는 계정과목 코드
error_journal.loc[0, "debit_account_1"] = 9999

# 2번 거래: 기준표에 없는 거래처 코드
error_journal.loc[1, "partner_code"] = "V999"

# 3번 거래: 기준표에 없는 부서 코드
error_journal.loc[2, "department_code"] = "D999"

# 4번 거래: 차변 금액을 변경해 차변·대변 불일치 발생
error_journal.loc[3, "debit_amount_1"] += 10_000

# 5번 거래: 부가세를 정상 금액과 다르게 변경
error_journal.loc[4, "vat_amount"] = 100_000

# 6번 거래: 필수 증빙번호 삭제
error_journal.loc[5, "evidence_no"] = None

# 7번 거래: 앞선 거래와 동일한 증빙번호를 입력
error_journal.loc[6, "evidence_no"] = "TAX-202608-004"

# 8번 거래: 분석 대상인 8월을 벗어난 거래일자
error_journal.loc[7, "transaction_date"] = pd.Timestamp("2026-09-05")

# 9번 거래: 필수 적요를 빈 문자열로 변경
error_journal.loc[8, "description"] = ""

# 10번 거래: 전표 합계와 다른 총금액 입력
error_journal.loc[9, "total_amount"] = 5_400_000

# 오류가 삽입된 주요 열 확인
error_journal[
    [
        "voucher_id",
        "transaction_date",
        "department_code",
        "partner_code",
        "evidence_no",
        "description",
        "total_amount"
    ]
]

,voucher_id,transaction_date,department_code,partner_code,evidence_no,description,total_amount
0,JV202608001,2026-08-01,D002,V001,TAX-202608-001,고무 원재료 외상 매입,5500000
1,JV202608002,2026-08-02,D003,V999,CARD-202608-001,생산 작업용 장갑 구매,220000
2,JV202608003,2026-08-03,D999,V004,TAX-202608-002,제품 포장재 외상 구매,880000
3,JV202608004,2026-08-05,D007,V005,TAX-202608-003,제품 납품 운송비,660000
4,JV202608005,2026-08-08,D003,V006,TAX-202608-004,공장 전기요금,1650000
5,JV202608006,2026-08-10,D002,V007,None,생산설비 외상 구매,22000000
6,JV202608007,2026-08-12,D001,V008,TAX-202608-004,회계 자문 수수료,1100000
7,JV202608008,2026-09-05,D006,C001,SALE-202608-001,자동차용 고무부품 외상 판매,13200000
8,JV202608009,2026-08-18,D006,C002,SALE-202608-002,,8800000
9,JV202608010,2026-08-20,D001,V001,BANK-202608-001,원재료 외상매입금 지급,5400000


## 3. 삽입한 오류 정답표 생성

In [3]:
# 각 거래에 의도적으로 삽입한 오류의 정답 정보
expected_errors = [
    {
        "voucher_id": "JV202608001",
        "column": "debit_account_1",
        "error_type": "존재하지 않는 계정과목"
    },
    {
        "voucher_id": "JV202608002",
        "column": "partner_code",
        "error_type": "존재하지 않는 거래처"
    },
    {
        "voucher_id": "JV202608003",
        "column": "department_code",
        "error_type": "존재하지 않는 부서"
    },
    {
        "voucher_id": "JV202608004",
        "column": "debit_amount_1",
        "error_type": "차변·대변 불일치"
    },
    {
        "voucher_id": "JV202608005",
        "column": "vat_amount",
        "error_type": "부가세 계산 오류"
    },
    {
        "voucher_id": "JV202608006",
        "column": "evidence_no",
        "error_type": "필수값 누락"
    },
    {
        "voucher_id": "JV202608007",
        "column": "evidence_no",
        "error_type": "증빙번호 중복"
    },
    {
        "voucher_id": "JV202608008",
        "column": "transaction_date",
        "error_type": "회계기간 이탈"
    },
    {
        "voucher_id": "JV202608009",
        "column": "description",
        "error_type": "필수값 누락"
    },
    {
        "voucher_id": "JV202608010",
        "column": "total_amount",
        "error_type": "전표 합계 불일치"
    }
]

expected_errors_df = pd.DataFrame(expected_errors)

expected_errors_df

,voucher_id,column,error_type
0,JV202608001,debit_account_1,존재하지 않는 계정과목
1,JV202608002,partner_code,존재하지 않는 거래처
2,JV202608003,department_code,존재하지 않는 부서
3,JV202608004,debit_amount_1,차변·대변 불일치
4,JV202608005,vat_amount,부가세 계산 오류
5,JV202608006,evidence_no,필수값 누락
6,JV202608007,evidence_no,증빙번호 중복
7,JV202608008,transaction_date,회계기간 이탈
8,JV202608009,description,필수값 누락
9,JV202608010,total_amount,전표 합계 불일치


## 4. 오류 분개장과 정답표 저장

In [4]:
# 오류가 포함된 분개장을 저장할 경로
error_journal_path = (
    project_root
    / "data"
    / "raw"
    / "journal_with_errors.xlsx"
)

# 의도적으로 삽입한 오류의 정답표를 저장할 경로
expected_output_path = (
    project_root
    / "data"
    / "expected"
    / "injected_errors.csv"
)

# 오류 분개장을 엑셀 파일로 저장
error_journal.to_excel(
    error_journal_path,
    index=False,
    sheet_name="분개장"
)

# 정답표는 한글이 깨지지 않도록 UTF-8 형식의 CSV로 저장
expected_errors_df.to_csv(
    expected_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("오류 분개장 저장 완료:", error_journal_path.exists())
print("정답표 저장 완료:", expected_output_path.exists())
print("저장된 거래 수:", len(error_journal))
print("저장된 오류 정답 수:", len(expected_errors_df))

오류 분개장 저장 완료: True
정답표 저장 완료: True
저장된 거래 수: 10
저장된 오류 정답 수: 10
